# Residual Stream Information

Information-theoretic analysis of the residual stream:
prediction entropy, information added per layer, KL convergence,
and mutual information proxies.

## Why This Matters

Residual stream stream information characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_information import (
    residual_prediction_entropy, information_added_per_layer,
    residual_kl_from_final, residual_mutual_information_proxy,
    residual_information_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Prediction Entropy Through Layers

How certainty builds as information flows through the network.

In [ ]:
result = residual_prediction_entropy(model, tokens, position=-1)
print(f"Final entropy: {result['final_entropy']:.4f}")
print(f"Entropy range: {result['entropy_range']:.4f}")
for p in result['per_layer']:
    bar = '█' * int(p['entropy'])
    print(f"  Layer {p['layer']}: H={p['entropy']:.4f} {bar}")

## Information Added Per Layer

Which layers contribute the most useful information?

In [ ]:
result = information_added_per_layer(model, tokens, position=-1)
print(f"Total information added: {result['total_info']:.4f}")
for p in result['per_layer']:
    flag = ' [INFORMATIVE]' if p['is_informative'] else ''
    sign = '+' if p['info_added'] > 0 else ''
    print(f"  Layer {p['layer']}: ΔH={sign}{p['info_added']:.4f} (H={p['entropy']:.4f}){flag}")

## KL Divergence from Final

How quickly do predictions converge to the final answer?

In [ ]:
result = residual_kl_from_final(model, tokens, position=-1)
print(f"Convergence layer: {result['convergence_layer']}")
print(f"Initial KL: {result['initial_kl']:.4f}")
for p in result['per_layer']:
    bar = '█' * int(p['kl_from_final'] * 5)
    print(f"  Layer {p['layer']}: KL={p['kl_from_final']:.4f} {bar}")

## Mutual Information Proxy

Logit variance as a proxy for information content.

In [ ]:
for layer in range(4):
    result = residual_mutual_information_proxy(model, tokens, layer=layer)
    print(f"  Layer {layer}: var={result['logit_variance']:.4f}, "
          f"mag={result['logit_mean_magnitude']:.4f}")

## Information Summary

In [ ]:
result = residual_information_summary(model, tokens, position=-1)
for k, v in result.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")